In [ ]:
import numpy as np
import pandas as pd

### Load datasets for modeling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

premodel_core = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Core_Final.csv")
premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")

Mounted at /content/drive


/tmp/ipython-input-854/274579847.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")


### Quick dataset checks

In [ ]:
print("Core Shape:", premodel_core.shape)
print("Gust Shape:", premodel_gust.shape)
print("Core Class Balance Check:", premodel_core["High_Intensity"].value_counts(normalize=True))
print("Gust Class Balance Check:", premodel_gust["High_Intensity"].value_counts(normalize=True))

Core Shape: (1563080, 21)
Gust Shape: (1011445, 25)
Core Class Balance Check: High_Intensity
0    0.900896
1    0.099104
Name: proportion, dtype: float64
Gust Class Balance Check: High_Intensity
0    0.886772
1    0.113228
Name: proportion, dtype: float64


In [ ]:
premodel_core["LC_Type1"] = premodel_core["LC_Type1"].astype("category")
premodel_gust["LC_Type1"] = premodel_gust["LC_Type1"].astype("category")

print(premodel_core.dtypes)
print(premodel_gust.dtypes)


High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
LC_Type1                        category
Month                              int64
Day_of_Year                        int64
Latitude_Fire                    float64
Longitude_Fire                   float64
Elevation_m                      float64
FRP_max                          float64
Fire_ID                            int64
Climate_ID                        object
LC_Label                          object
Nearest_Core_Station_Dist_km     float64
Core_Dist_75km                      bool
Core_Dist_100km                     bool
Confidence_Ordered                 int64
DayNight                          object
Hour                               int64
Year                               int64
Province                          object
dtype: object
High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
Ma

# **1. LOGISTIC REGRESSION FOR CORE DATASET**

## **(1) Define Features and Response**

In [ ]:
# Core feature set for baseline model
core_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year",
    "LC_Type1"
]

X = premodel_core[core_features]
y = premodel_core["High_Intensity"]

### **(2) Define Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Startified split to ensure rare high-intensity class representation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

### **(3) Feature Adjustment Preprocessing**

In [ ]:
# Define feature types (numerical and categorical)
numeric_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year"
]

categorical_features = ["LC_Type1"]

### Construct preprocessing and model pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=123
)

pipeline = Pipeline(
    steps=[("preprocess", preprocess), ("model", log_reg)]
)

### **(4) Fit Model & Evaluate Performance**

In [ ]:
# Fit the model
pipeline.fit(X_train, y_train)

# Evaluate Performance
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC–AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.51      0.65    281634
           1       0.12      0.63      0.21     30982

    accuracy                           0.52    312616
   macro avg       0.53      0.57      0.43    312616
weighted avg       0.85      0.52      0.61    312616

ROC–AUC: 0.5961600677339053
Confusion Matrix:
 [[142478 139156]
 [ 11325  19657]]


### **(5) Check Model Coefficients**

In [ ]:
# Extract feature names after preprocessing
feature_names = (
    pipeline.named_steps["preprocess"]
    .get_feature_names_out()
)

coefficients = pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values("Coefficient", ascending=False)

coef_df.head(10), coef_df.tail(10)

(             Feature  Coefficient
 13   cat__LC_Type1_7     1.587311
 15   cat__LC_Type1_9     0.958912
 14   cat__LC_Type1_8     0.806973
 16  cat__LC_Type1_10     0.687114
 7    cat__LC_Type1_1     0.494067
 5         num__Month     0.432962
 10   cat__LC_Type1_4     0.400318
 11   cat__LC_Type1_5     0.264833
 17  cat__LC_Type1_11     0.243507
 9    cat__LC_Type1_3     0.107214,
                  Feature  Coefficient
 21      cat__LC_Type1_15     0.003224
 12       cat__LC_Type1_6    -0.001371
 2     num__Latitude_Fire    -0.008402
 1   num__Total_Precip_mm    -0.125866
 23      cat__LC_Type1_17    -0.330109
 18      cat__LC_Type1_12    -0.369943
 6       num__Day_of_Year    -0.575249
 20      cat__LC_Type1_14    -0.640042
 22      cat__LC_Type1_16    -2.276173
 19      cat__LC_Type1_13    -2.656670)

# **2. LOGISTIC REGRESSION FOR GUST DATASET**

## **(1) Define Features and Response**

In [ ]:
# Gust feature set for baseline model
gust_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Max_Gust_Speed_kmh",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year",
    "LC_Type1"
]

X = premodel_gust[gust_features]
y = premodel_gust["High_Intensity"]

### **(2) Define Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Startified split to ensure rare high-intensity class representation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

### **(3) Feature Adjustment Preprocessing**

In [ ]:
# Define feature types (numerical and categorical)
numeric_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Max_Gust_Speed_kmh",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year"
]

categorical_features = ["LC_Type1"]

### Construct preprocessing and model pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=123
)

pipeline = Pipeline(
    steps=[("preprocess", preprocess), ("model", log_reg)]
)

### **(4) Fit Model & Evaluate Performance**

In [ ]:
# Fit the model
pipeline.fit(X_train, y_train)

# Evaluate Performance
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC–AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.52      0.66    179384
           1       0.14      0.63      0.23     22905

    accuracy                           0.53    202289
   macro avg       0.53      0.58      0.45    202289
weighted avg       0.83      0.53      0.62    202289

ROC–AUC: 0.6022121563403529
Confusion Matrix:
 [[93449 85935]
 [ 8459 14446]]


### **(5) Check Model Coefficients**

In [ ]:
# Extract feature names after preprocessing
feature_names = (
    pipeline.named_steps["preprocess"]
    .get_feature_names_out()
)

coefficients = pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values("Coefficient", ascending=False)

coef_df.head(10), coef_df.tail(10)

(                    Feature  Coefficient
 13          cat__LC_Type1_7     1.627521
 15          cat__LC_Type1_9     0.850868
 14          cat__LC_Type1_8     0.778309
 16         cat__LC_Type1_10     0.701859
 8           cat__LC_Type1_1     0.471620
 11          cat__LC_Type1_4     0.386296
 9           cat__LC_Type1_2     0.293795
 12          cat__LC_Type1_5     0.220337
 2   num__Max_Gust_Speed_kmh     0.191517
 17         cat__LC_Type1_11     0.164927,
                  Feature  Coefficient
 0       num__Mean_Temp_C     0.067052
 5       num__Elevation_m     0.030020
 21      cat__LC_Type1_15    -0.027196
 7       num__Day_of_Year    -0.142689
 1   num__Total_Precip_mm    -0.156772
 18      cat__LC_Type1_12    -0.389735
 23      cat__LC_Type1_17    -0.405174
 20      cat__LC_Type1_14    -0.537075
 22      cat__LC_Type1_16    -2.319816
 19      cat__LC_Type1_13    -2.562327)